# 第 23 课｜价差/流动性/冲击成本

> **数据身份**：固定随机种子生成的模拟教学数据，不是真实证券行情。  
> **环境**：`pkm-quant` · Python 3.11 · 默认离线运行。  
> **课程原始材料**：`23_youtube-PQ62jZ8V_QQ/index.html`  
> **所属模块**：P1-M5 · 市场微观结构与执行

## 本节量化问题

理论信号收益经过真实成交后还剩多少？

## 准确定义与策略报告位置

总交易成本包括显性费用、买卖价差、滑点和市场冲击；规模越大通常需要穿透更多档位。

- **核心关系**：成本后收益=毛收益-费用-价差成本-滑点-市场冲击；冲击通常随成交规模非线性增加。
- **写入报告**：费用、滑点、冲击、换手和容量
- **P2 衔接**：P2 第 36、40、46 课：成本后净值和报告流水线。


## 运行环境与数据契约

先确认项目、数据目录和数据身份。后续联网数据源必须通过适配层进入，不能改变本 Notebook 的研究计算口径。

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "fixtures"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_SEED = 365
print("项目:", ROOT)
print("数据:", DATA_DIR)
print("数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情")


项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations
数据: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/data/fixtures
数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情


## 按档位穿透计算买入均价

市价买入从最低卖价开始消耗数量。成交均价与决策时中间价的差构成实现滑点的一部分。

In [2]:
book = pd.read_csv(DATA_DIR / "order_book.csv")
asks = book.loc[book["side"] == "ask"].sort_values("price")
best_bid = book.loc[book["side"] == "bid", "price"].max()
best_ask = asks["price"].min()
mid = (best_bid + best_ask) / 2.0

def market_buy_vwap(quantity):
    remaining = quantity
    cost = 0.0
    filled = 0
    for row in asks.itertuples():
        take = min(remaining, int(row.quantity))
        cost += take * row.price
        filled += take
        remaining -= take
        if remaining == 0:
            break
    if remaining:
        raise ValueError("可见卖盘深度不足")
    return cost / filled

sizes = [100, 600, 1_500]
rows = []
for size in sizes:
    vwap = market_buy_vwap(size)
    rows.append({"quantity": size, "vwap": vwap, "slippage_bps_vs_mid": (vwap / mid - 1.0) * 10_000})
display(pd.DataFrame(rows))

,quantity,vwap,slippage_bps_vs_mid
0,100,100.010000,1.000000
1,600,100.015500,1.550000
2,1500,100.025733,2.573333


## 方法边界与阶段验收

固定基点成本无法描述容量和状态变化；历史收盘价成交是常见乐观假设。

完成本课后应能：

1. 不看课程材料准确解释核心概念和公式符号；
2. 用小样本手工核对 Notebook 的关键结果；
3. 说明结果能支持什么判断、不能支持什么判断；
4. 指出它在策略报告中的位置，以及仍缺少的数据、成本或稳健性证据。
